# 문제 4~5~6: Parquet 저장, Window Function, Broadcast Join

한 Colab 노트북 안에서 문제 4, 문제 5, 문제 6 코드를 별도 셀로 실행합니다. 문제 4에서 실제 AWS S3에 저장한 Parquet 경로를 문제 5와 문제 6에서 이어서 사용합니다.

In [ ]:
# Colab 실행 환경 준비
%pip install -q pyspark==3.5.3 boto3

## 공통 준비 코드

In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

import boto3
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import avg, broadcast, col, count, rank, round, sum, to_date, when
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType, TimestampType


AWS_ACCESS_KEY_ID = ""
AWS_SECRET_ACCESS_KEY = ""
AWS_DEFAULT_REGION = "ap-northeast-2"
S3_BUCKET = "YOUR_BUCKET_NAME"
SENSOR_PARQUET_PREFIX = "sensor/parquet/"
FACTORY_ERROR_PREFIX = "sensor/factory_error_rate/"

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(message)s",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("problem4_5_6_sensor_pipeline")

if S3_BUCKET == "YOUR_BUCKET_NAME":
    raise ValueError("S3_BUCKET 값을 실제 AWS S3 버킷명으로 변경하세요.")

s3 = boto3.Session(region_name=AWS_DEFAULT_REGION).client("s3")
q4_s3_path = f"s3a://{S3_BUCKET}/{SENSOR_PARQUET_PREFIX}"
local_parquet_dir = Path("/content/sensor_parquet_output") if Path("/content").exists() else Path("sensor_parquet_output")
restored_parquet_dir = Path("/content/sensor_parquet_from_s3") if Path("/content").exists() else Path("sensor_parquet_from_s3")

sensor_schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("vibration", DoubleType(), True),
    StructField("pressure", DoubleType(), True),
    StructField("status", StringType(), True),
])

equipment_schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("factory", StringType(), True),
    StructField("line", StringType(), True),
])

spark = (
    SparkSession.builder
    .appName("DE_5_week_problem4_5_6_sensor_pipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


def find_csv(filename):
    candidates = [Path(filename), Path("/content") / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    try:
        from google.colab import files

        print(f"{filename} 파일을 업로드하세요.")
        uploaded = files.upload()
        if filename not in uploaded:
            raise FileNotFoundError(f"업로드 파일명은 {filename} 이어야 합니다.")
        return Path(filename)
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(f"{filename} 파일을 현재 폴더에 두고 다시 실행하세요.") from exc


def clear_s3_prefix(prefix):
    objects = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix).get("Contents", [])
    if objects:
        s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": [{"Key": obj["Key"]} for obj in objects]})


def upload_directory_to_s3(local_dir, prefix):
    clear_s3_prefix(prefix)
    for file_path in Path(local_dir).rglob("*"):
        if file_path.is_file():
            key = prefix + file_path.relative_to(local_dir).as_posix()
            s3.put_object(Bucket=S3_BUCKET, Key=key, Body=file_path.read_bytes())


def restore_s3_prefix_to_directory(prefix, local_dir):
    local_dir = Path(local_dir)
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    objects = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix).get("Contents", [])
    if not objects:
        raise RuntimeError(f"s3://{S3_BUCKET}/{prefix} 경로에 파일이 없습니다. 문제 4를 먼저 실행하세요.")

    for obj in objects:
        key = obj["Key"]
        target = local_dir / Path(key).relative_to(prefix)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read())
    return local_dir


def print_s3_objects(prefix):
    objects = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix).get("Contents", [])
    print("S3 저장 확인: Key / LastModified / Size")
    for obj in objects:
        print(f"Key: {obj['Key']}")
        print(f"LastModified: {obj['LastModified']}")
        print(f"Size: {obj['Size']} bytes")

## 문제 4 코드: CSV 스키마 지정, Parquet 변환, 실제 S3 저장

In [ ]:
try:
    csv_path = find_csv("sensor_logs.csv")
    df = spark.read.schema(sensor_schema).option("header", "true").csv(str(csv_path))

    print("스키마 출력")
    df.printSchema()
    df.show(5, truncate=False)

    df_with_date = df.withColumn("date", to_date(col("timestamp")))
    df_with_date.select("machine_id", "timestamp", "date").show(3, truncate=False)

    if local_parquet_dir.exists():
        shutil.rmtree(local_parquet_dir)
    df_with_date.write.mode("overwrite").partitionBy("machine_id", "date").parquet(str(local_parquet_dir))
    upload_directory_to_s3(local_parquet_dir, SENSOR_PARQUET_PREFIX)

    print(f"Parquet 저장 대상: {q4_s3_path}")
    print_s3_objects(SENSOR_PARQUET_PREFIX)

    first_count = spark.read.parquet(str(local_parquet_dir)).count()
    second_count = spark.read.parquet(str(local_parquet_dir)).count()
    print(f"1회차 row count: {first_count}")
    print(f"2회차 row count: {second_count}")
    print(f"동일 여부: {first_count == second_count}")
    logger.info("문제 4 처리 완료")

except Exception as exc:
    logger.exception("문제 4 처리 실패: %s", exc)
    raise

## 문제 5 코드: 문제 4의 S3 Parquet 로딩 및 Window Function

In [ ]:
try:
    restore_s3_prefix_to_directory(SENSOR_PARQUET_PREFIX, restored_parquet_dir)
    print(f"S3 Parquet 로딩 경로: {q4_s3_path}")

    df_logs = spark.read.parquet(str(restored_parquet_dir))
    print(f"로딩 row count: {df_logs.count()}")

    time_window = Window.partitionBy("machine_id").orderBy("timestamp")
    moving_window = time_window.rowsBetween(-9, 0)
    vibration_window = Window.partitionBy("machine_id").orderBy(col("vibration").desc())
    cumulative_window = time_window.rowsBetween(Window.unboundedPreceding, 0)

    result_df = (
        df_logs
        .withColumn("temp_moving_avg", avg("temperature").over(moving_window))
        .withColumn("vibration_rank", rank().over(vibration_window))
        .withColumn("error_cumsum", sum(when(col("status") == "오류", 1).otherwise(0)).over(cumulative_window))
    )

    anomaly_df = result_df.where((col("temp_moving_avg") > 85) | (col("vibration_rank") <= 5))
    print(f"이상 감지 건수: {anomaly_df.count()}")
    anomaly_df.select(
        "machine_id", "timestamp", "status", "temp_moving_avg", "vibration_rank", "error_cumsum"
    ).show(20, truncate=False)
    logger.info("문제 5 처리 완료")

except Exception as exc:
    logger.exception("문제 5 처리 실패: %s", exc)
    raise

## 문제 6 코드: Broadcast Join 및 공장별 불량률 집계

In [ ]:
try:
    restore_s3_prefix_to_directory(SENSOR_PARQUET_PREFIX, restored_parquet_dir)
    print(f"S3 Parquet 로딩 경로: {q4_s3_path}")

    df_logs = spark.read.parquet(str(restored_parquet_dir))
    equipment_path = find_csv("equipment.csv")
    df_equip = spark.read.schema(equipment_schema).option("header", "true").csv(str(equipment_path))
    df_equip.show(5, truncate=False)

    print(f"join 전 파티션 수: {df_logs.rdd.getNumPartitions()}")
    df_joined = df_logs.join(broadcast(df_equip), "machine_id")
    print(f"join 후 파티션 수: {df_joined.rdd.getNumPartitions()}")

    df_repartitioned = df_joined.repartition(4)
    print(f"repartition(4) 후 파티션 수: {df_repartitioned.rdd.getNumPartitions()}")

    factory_error_df = (
        df_repartitioned
        .groupBy("factory")
        .agg(
            count("*").alias("total_logs"),
            sum(when(col("status") == "오류", 1).otherwise(0)).alias("error_count"),
        )
        .withColumn("error_rate_pct", round(col("error_count") / col("total_logs") * 100, 2))
        .orderBy("factory")
    )

    print("공장별 불량률 집계")
    factory_error_df.show(truncate=False)

    result_output_dir = Path("/content/factory_error_rate_output") if Path("/content").exists() else Path("factory_error_rate_output")
    if result_output_dir.exists():
        shutil.rmtree(result_output_dir)
    factory_error_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(str(result_output_dir))
    upload_directory_to_s3(result_output_dir, FACTORY_ERROR_PREFIX)

    print(f"공장별 불량률 CSV 저장 대상: s3://{S3_BUCKET}/{FACTORY_ERROR_PREFIX}")
    print_s3_objects(FACTORY_ERROR_PREFIX)
    print("broadcast를 적용한 이유: equipment는 설비 수가 적은 소규모 마스터 테이블이므로 각 executor에 복사해 shuffle 비용을 줄일 수 있습니다.")
    logger.info("문제 6 처리 완료")

except Exception as exc:
    logger.exception("문제 6 처리 실패: %s", exc)
    raise